# 01 Import and Audit\nPurpose: Load raw dataset, validate schema, and produce baseline quality audit artifacts.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\n\nROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nRAW_PATH = ROOT / 'data' / 'raw' / 'Loan_default.csv'\nPROFILE_OUT = ROOT / 'data' / 'processed' / '01_column_profile.csv'\nAUDIT_OUT = ROOT / 'docs' / 'phase_04_audit_summary.md'\nROOT

In [ ]:
df = pd.read_csv(RAW_PATH)\ndf.head()

In [ ]:
print('shape:', df.shape)\nprint('duplicate rows:', df.duplicated().sum())\nif 'LoanID' in df.columns:\n    print('duplicate LoanID:', df['LoanID'].duplicated().sum())\nif 'Default' in df.columns:\n    print('default rate %:', round(df['Default'].mean() * 100, 3))

In [ ]:
profile = pd.DataFrame({\n    'column': df.columns,\n    'dtype': [str(t) for t in df.dtypes],\n    'missing_count': df.isna().sum().values,\n    'missing_pct': (df.isna().mean() * 100).round(4).values,\n    'unique_count': [df[c].nunique(dropna=True) for c in df.columns],\n})\nprofile.sort_values(['missing_pct', 'unique_count'], ascending=[False, False]).head(20)

In [ ]:
PROFILE_OUT.parent.mkdir(parents=True, exist_ok=True)\nprofile.to_csv(PROFILE_OUT, index=False)\nsummary_lines = [\n    '# Phase 4 Audit Summary',\n    f'- Rows: {df.shape[0]:,}',\n    f'- Columns: {df.shape[1]}',\n    f'- Duplicate Rows: {int(df.duplicated().sum())}',\n    f"- Default Rate (%): {round(df['Default'].mean() * 100, 3) if 'Default' in df.columns else 'N/A'}",\n]\nAUDIT_OUT.write_text('\n'.join(summary_lines), encoding='utf-8')\nprint('saved:', PROFILE_OUT)\nprint('saved:', AUDIT_OUT)